---
# Python API
One `RunModel` instance; call stages in order. You can stop, inspect files, and continue — but note stages 3→5 share paths in memory, so within a *fresh* session start from the stage whose inputs already exist on disk (each stage re-derives what it needs).

## Scenario Configuration [Interactive]

In [ ]:
import bcnexus.utils as utils
(scenarios_box, storage_tg, hg, nc, ts_label,
 solver_bg, compiler_bg, threads_bg) = utils.build_scenario_ui() # note the int() cast

- Ingest values from the UI

In [ ]:
scenarios = utils.get_selected_scenarios(scenarios_box)   # or scenarios_box.value
STORAGE_ALGO  = storage_tg.value # "Kotzur"           # Kotzur | Niet
HOUR_GROUPING = hg.value # 3
N_CLUSTERS    = nc.value # 4
SOLVER        = solver_bg.value.lower()          # gurobi | cbc
COMPILER      = compiler_bg.value.lower()        # cplex | gurobi | xpress | cbc
THREADS       = threads_bg.value # 16
INCLUDE_LIVESTOCK = False  

- Initiate the `RunModel` object

In [ ]:
from bcnexus.clews.runner import RunModel

# X Full Workflow with Scenario pipeline

In [ ]:
import yaml
cfg = yaml.safe_load(open('config/scenarios_bcnexus.yaml'))['SCENARIOS']
missing = [s for s in scenarios if s not in cfg]
assert not missing, f"Not in scenarios_bcnexus.yaml: {missing}"
print(f"Planned: {len(scenarios)} scenarios x {storage_tg.value} x "
      f"{(24//hg.value)*nc.value} timeslices, solver={solver_bg.value}")

In [ ]:
import time
import traceback
import pandas as pd

results_log = []

for scene in scenarios:
    utils.print_banner(f"Running scenario: {scene}")
    t0 = time.time()
    try:
        m = RunModel(run_scenario=scene,
                     storage_algorithm=storage_tg.value,
                     clustering_attributes={'hour_grouping': hg.value,
                                            'n_clusters': nc.value})
        m.run(build=True, include_livestock=False,
              solver_name=solver_bg.value.lower(),
              threads=int(threads_bg.value))
        status, note = "OK", ""
        utils.print_banner(f"Finished scenario: {scene}")

    except KeyboardInterrupt:                 # let Ctrl-C stop the whole batch
        print("Batch interrupted by user.")
        raise

    except Exception as e:
        status, note = "FAILED", f"{type(e).__name__}: {e}"
        utils.print_error(f"X {scene} failed — continuing with next scenario")
        traceback.print_exc()                 # full trace stays in the output

    results_log.append({"scenario": scene,
                        "status": status,
                        "minutes": round((time.time() - t0) / 60, 1),
                        "note": note})

summary = pd.DataFrame(results_log)
display(summary)
print(f"{(summary.status=='OK').sum()} succeeded, "
      f"{(summary.status=='FAILED').sum()} failed, of {len(summary)}")

2026-07-19 11:02:57,170 - INFO -   21   6.53289730e+07 -8.72003621e+07  5.83e+02 1.70e+00  6.02e+02    41s
2026-07-19 11:02:59,188 - INFO -   22   5.81923846e+07 -6.43596784e+07  2.10e+02 1.35e+00  4.67e+02    43s
2026-07-19 11:03:01,309 - INFO -   23   5.34506248e+07 -4.61514091e+07  1.26e+02 1.08e+00  3.68e+02    45s
2026-07-19 11:03:03,067 - INFO -   24   4.97470985e+07 -3.34029231e+07  1.05e+02 8.91e-01  2.99e+02    46s
2026-07-19 11:03:04,822 - INFO -   25   4.71874738e+07 -2.05942849e+07  8.90e+01 7.12e-01  2.38e+02    48s
2026-07-19 11:03:06,625 - INFO -   26   4.49700152e+07 -1.24162175e+07  7.35e+01 6.02e-01  1.97e+02    50s
2026-07-19 11:03:08,355 - INFO -   27   4.34059299e+07 -1.39636329e+06  6.15e+01 4.58e-01  1.50e+02    52s
2026-07-19 11:03:10,101 - INFO -   28   4.19137416e+07  8.28982968e+06  4.85e+01 3.35e-01  1.10e+02    53s
2026-07-19 11:03:12,193 - INFO -   29   4.09096448e+07  1.64808459e+07  3.92e+01 2.35e-01  7.85e+01    56s
2026-07-19 11:03:15,151 - INFO -   30

  └> --------------------------------------------------
  └─> Model run completed. Please check the log for detailed report.
  └> Collecting constraints reports...
  └> Summary saved to results/clews/Model_Kotzur_Base_CNZ/24ts/constraints_summary.txt
 └> Initiating otoole interface to extract results; input csvs : data/clews_data/clews_build_data/Model_Kotzur/storage_case_input_csvs , config file: models/model_Kotzur/otoole_config_Kotzur.yaml
  └> Result extraction completed and saved to results/clews/Model_Kotzur_Base_CNZ/24ts/result_csvs_gurobi
 └> Initiating otoole interface to extract results; input csvs : data/clews_data/clews_build_data/Model_Kotzur/storage_case_input_csvs , config file: models/model_Kotzur/otoole_config_Kotzur.yaml
  └> Result extraction completed and saved to results/clews/Model_Kotzur_Base_CNZ/24ts/result_csvs_gurobi
└> Loaded 29 CSV files from results/clews/Model_Kotzur_Base_CNZ/24ts/result_csvs_gurobi
 └ ❌ > X Base_CNZ failed — continuing with next scenario


Traceback (most recent call last):
  File "/tmp/ipykernel_3558709/2204212983.py", line 15, in <module>
    m.run(build=True, include_livestock=False,
  File "/local-scratch/localhome/mei3/eliasinul/work/BC_Nexus/bcnexus/clews/runner.py", line 698, in run
    plotter.get_plots(**plotter_args)
  File "/local-scratch/localhome/mei3/eliasinul/work/BC_Nexus/bcnexus/plots.py", line 167, in get_plots
    if not plots_save_to.exists():
           ^^^^^^^^^^^^^^^^^^^^
AttributeError: 'str' object has no attribute 'exists'


********************************
Running scenario: Base_CNZ_noCCS
********************************
└> Initiated CLEWs Model Runner for 'Base_CNZ_noCCS' scenario with 'Kotzur' storage algorithm
 └>  Input CSVS set to: data/clews_data/clews_build_data/Model_Kotzur/storage_case_input_csvs
____________________________________________________________________________________________________
     CLEWs Model Builder
____________________________________________________________________________________________________
************************
Scenario: Base_CNZ_noCCS
************************
ℹ️  Using configuration file at: config/scenarios_bcnexus.yaml
ℹ️  Clustering attributes: {'hour_grouping': 4, 'n_clusters': 4}
ℹ️  Storage Algorithm: Kotzur
 └> Extracting class attributes e.g. directories, static values/ranges, constants etc.
bcnexus.attributes_parser | CLEWs snapshot configuration: start_year=2021, last_year=2050
ℹ️  bcnexus.attributes_parser | BCNexus CLEWs model is structured as SINGLE 

2026-07-19 11:08:04,163 - INFO - Read LP format model from file data/clews_data/clews_build_data/Model_Kotzur/Base_CNZ_noCCS/Base_CNZ_noCCS.lp


...

2026-07-19 11:13:31,286 - INFO - Reading time = 2.68 seconds


cost: 707325 rows, 767429 columns, 7600856 nonzeros


2026-07-19 11:13:31,287 - INFO - cost: 707325 rows, 767429 columns, 7600856 nonzeros


Set parameter LogToConsole to value 0


2026-07-19 11:13:31,287 - INFO - Set parameter LogToConsole to value 0
2026-07-19 11:13:31,287 - INFO - Set parameter Method to value 2
2026-07-19 11:13:31,287 - INFO - Set parameter Threads to value 24
2026-07-19 11:13:31,287 - INFO - Set parameter NumericFocus to value 2
2026-07-19 11:13:31,287 - INFO - Set parameter Crossover to value 0
2026-07-19 11:13:31,287 - INFO - Set parameter ScaleFlag to value 2
2026-07-19 11:13:31,287 - INFO - Set parameter BarHomogeneous to value 1
2026-07-19 11:13:31,288 - INFO - Set parameter LogFile to value "results/clews/Model_Kotzur_CNZ_LIMITED_CO2/24ts/gurobi.log"
2026-07-19 11:13:31,288 - INFO - Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "Ubuntu 20.04.6 LTS")
2026-07-19 11:13:31,288 - INFO - 
2026-07-19 11:13:31,288 - INFO - CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
2026-07-19 11:13:31,288 - INFO - Thread count: 32 physical cores, 32 logical processors, using up to 24 threads
2026-07-19 11:13:

# X  Inspection - individual stage, per scenario

 BCNexus — Stage-wise Workflow explained
----
**Author:** Md Eliasinul Islam

Run the CLEWs pipeline **one stage at a time**, via the Python API or the CLI — both drive the same `RunModel.stage_*` methods, so results are identical.

| Stage | Python | CLI |
|---|---|---|
| 1. Build SETs/params | `m.stage_build(include_livestock=INCLUDE_LIVESTOCK)` | `python -m bcnexus.stages build ...` |
| 2. Scenario overrides | `m.stage_scenario()` | `... scenario-overlay ...` |
| 3. otoole datafile | `m.stage_datafile()` | `... datafile ...` |
| 4. LP file (glpsol) | `m.stage_lp()` | `... lp ...` |
| 5. Solve (+duals/reports) | `m.stage_solve()` | `... solve ...` |
| 6. Result CSVs (otoole) | `m.stage_results()` | `... results ...` |

Why stages: after a crash (or a config tweak) you re-run only the affected step — e.g. edit `scenarios_bcnexus.yaml`, then re-run stages 2→6 without rebuilding.

> Run this notebook from the **repo root** (where `config/` and `data/` live).

## Set a specific scenario

- See all scenarios configured

In [ ]:
scenarios

- Select a specific scenario

In [ ]:
scene=scenarios[0]

- Initiate the `RunModel` object with this scenario setup

In [ ]:
m = RunModel(run_scenario=scene, storage_algorithm=storage_tg.value,
                 clustering_attributes={'hour_grouping': hg.value, 'n_clusters': nc.value})

## Stage 1 — CLEWs builder: SETs, params, temporal profiles, storage schema

In [ ]:
# built_csvs = m.stage_build(include_livestock=INCLUDE_LIVESTOCK)

## Stage 2 — scenario overrides from config/scenarios_bcnexus.yaml

In [ ]:
# m.stage_scenario()

## Stage 3 — otoole datafile + model file

In [ ]:
# data_file, model_file = m.stage_datafile()
# print(data_file, model_file, sep="\n")

## Stage 4 — LP file via glpsol

- Planned to try MOSOX here

In [ ]:
# lp_file = m.stage_lp()
# lp_file

## Stage 5 — solve (writes .sol + duals, constraints summary, ELCB02 shadow price)

In [ ]:
# sol = m.stage_solve(solver_name=SOLVER, threads=THREADS)
# sol   # None => not optimal; check the solver log in the run dir

## Stage 6 — extract result CSVs via otoole

- <span style="color: grey;"> Check Results @ `results\clews\<Model_<storage_algorithm>_<scenario>`</span>

In [ ]:
results_dir = m.stage_results(solver_name=SOLVER)
results_dir  # => .../{N}ts/result_csvs_gurobi/

- <span style="color: grey;"> We can also use the `datapackage` module's `GetDataPackage` object to load load the results as an object.
- <span style="color: grey;"> This results pack could be used for __post-analysis, result exchange to other models, visualizations, scenario dashes__ etc.

- <span style="color: grey;">  Get `result_pack` as an instance of `GetDataPackage` object

In [ ]:
from bcnexus.clews.datapackage import GetDataPackage
result_pack=GetDataPackage(results_dir)

- <span style="color: grey;">  Apply `get_dataframe(<Result param name>)` method of the `result_pack` instance

## <span style="color: yellow;"> Plots </span>

- <span style="color: biege;">  Load the `plotter` module. 
  - <span style="color: grey;"> plotter.`get_plots()` function returns a dictionary of plots. You can review the plots in this notebook from that dictionary. The function also save the plots as html to 'vis/bccm/< scenario > '

In [ ]:
results_dir.parent

In [ ]:
from pathlib import Path
vis_dir = Path(str(results_dir).replace('results', 'vis', 1))

In [ ]:
results_dir.parent

In [ ]:
run_log=results_dir.parent/'runtime_memory_log.txt'
gurobi_log=results_dir.parent/'gurobi.log'
constraints_log=results_dir.parent/'constraints_summary.txt'

In [ ]:
from pathlib import Path
import bcnexus.plots as plotter

plotter_args=dict(
    nexus_scenario=scene,
    storage_algorithm=STORAGE_ALGO,
    timeslices=m.timeslices,
    results_csvs=results_dir,
    plots_save_to=Path('vis')
)
# plotter.main(**plotter_args)
nexus_plots_dict=plotter.get_plots(**plotter_args,
                              extra_info="Plots are under validation. Please consult to developer before citation") # enable `save_individual`=True for individual plots

- check what's inside the plot dict

In [ ]:
utils.print_banner("Plot categories")
for k1,v1 in nexus_plots_dict.items():
    if k1 not in ["Climate","Land","Energy","Water"]:
            utils.print_info(f"timeslices: {k1}")
    else:
        utils.print_update(level=1,message=k1)
        for k2,v2 in v1.items():
            utils.print_update(level=2,message=k2)
        

### Stable Plots

In [ ]:
nexus_plots_dict['Water']['hydro_water_energy']

In [ ]:
nexus_plots_dict['Energy']['power_generation_annual']

In [ ]:
nexus_plots_dict['Energy']['power_generation_timeslices'] 

### Dev. Plots
!! <span style="color: biege;"> Still under active development to include more comprehensive plots <span style="color: yellow;"> EL_20260719

#### Lands

In [ ]:
import bcnexus.vis.plot_Land as Lvis

In [ ]:
# Lvis.plot_land_area_by_crop (result_pack.get_df('ProductionByTechnologyAnnual'),scene)

-  The intensity line is diagnostic, and here it's telling you something is off. 0.08–0.10 BCM per 1000 km² converts to 80–100 mm of irrigation water per year — real irrigation applications run several hundred mm/yr. Combined with observation 1, the likely story: irrigated land techs in the model carry a low water input requirement (IAR), making irrigation nearly free, so the optimizer picks HI/II everywhere for the yield advantage. Cheap water in → total irrigation dominance out.
-  The plausibility check this demands. Statistics Canada puts BC's actually irrigated farmland at roughly 1–2 thousand km² — an order of magnitude below the ~27 kkm² the model shows. So this plot, doing exactly its job, is flagging that either the water-input coefficients on irrigated land techs or the yield differentials between regimes need re-examination before land–water results are cited. Check InputActivityRatio for LNDxxx HI/II techs → AGRWATBC1, against a crop-water benchmark (e.g., FAO crop water requirements, ~400–700 mm for most temperate crops).
This matters doubly for the Canada scale-up: the Prairies' irrigation question (Alberta's irrigation districts, Lake Diefenbaker expansion) is a live policy issue, and a model that irrigates everything for free would produce confidently wrong answers there. Worth logging as a calibration issue in the project notes — want me to add it to the plan document's verification items?

In [ ]:
Lvis.plot_irrigated_vs_rainfed (result_pack.get_df('ProductionByTechnologyAnnual'),scene)

In [ ]:
# Lvis.plot_landcover_change (result_pack.get_df('TotalCapacityAnnual'),scene)

In [ ]:
# Lvis.plot_cropland_change   (result_pack.get_df('TotalCapacityAnnual'),scene)

In [ ]:
# Lvis.plot_effective_yield (result_pack.get_df('ProductionByTechnologyAnnual'),scene)

In [ ]:
# Lvis.plot_forest_trajectory (result_pack.get_df('TotalCapacityAnnual'),scene)

In [ ]:
# Lvis.plot_cluster_crop_heatmap (result_pack.get_df('TotalAnnualTechnologyActivityByMode'), year=2040)


# modes = [m.strip().strip("'") for m in open("data/clews_data/SETs/ModeList.txt").read().split(",") if m.strip()]
# mode_names = {i + 1: name for i, name in enumerate(modes)}

# bymode = result_pack.get_df('TotalAnnualTechnologyActivityByMode')
# Lvis.plot_cluster_crop_heatmap(bymode, year=2035, mode_names=mode_names)

#### Climate

In [ ]:
import bcnexus.vis.plot_Climate as Cvis
# Cvis.plot_cumulative_emissions(result_pack.get_df('AnnualEmissions'), scene)

In [ ]:
# Cvis.plot_emissions_penalty_cost(result_pack.get_df('DiscountedTechnologyEmissionsPenalty'), scene)

In [ ]:
# Cvis.plot_sector_emission_intensity (result_pack.get_df('AnnualTechnologyEmission'),result_pack.get_df('ProductionByTechnologyAnnual'), scene)

In [ ]:
Cvis.plot_target_gap (result_pack.get_df('AnnualTechnologyEmission'), 0)

In [ ]:
import pandas as pd
from pathlib import Path


root = Path("results/clews")
runs = {
    "REF": pd.read_csv(root / "Model_Kotzur_Base_CNZ/32ts/result_csvs_gurobi/AnnualTechnologyEmission.csv"),
    "CO2 limit": pd.read_csv(root / "Model_Kotzur_CNZ_LIMITED_CO2/32ts/result_csvs_gurobi/AnnualTechnologyEmission.csv"),
    "CO2 penalty": pd.read_csv(root / "Model_Kotzur_CNZ_LIMITED_CO2_PTY/32ts/result_csvs_gurobi/AnnualTechnologyEmission.csv"),
}
fig = Cvis.plot_scenario_emission_wedges(runs, reference="REF")
fig.show()

#### Water

In [ ]:
import bcnexus.vis.plot_Water as Wvis

In [ ]:
# Wvis.plot_water_balance(result_pack.get_df('ProductionByTechnologyAnnual'), scene)

In [ ]:
# Wvis.plot_water_use_by_purpose(result_pack.get_df('ProductionByTechnologyAnnual'), scene)

In [ ]:
# Wvis.plot_reservoir_levels (result_pack.get_df('StorageLevelYearStart'), scene)

In [ ]:
Wvis.plot_hydro_water_energy  (result_pack.get_df('ProductionByTechnologyAnnual'),result_pack.get_df('StorageLevelYearStart'), scene)

#### Energy

In [ ]:
import bcnexus.vis.plot_Energy as Evis

In [ ]:
Evis.plot_system_cost_breakdown(result_pack.get_df('CapitalInvestment'), 
                                result_pack.get_df('AnnualFixedOperatingCost'), 
                                result_pack.get_df('AnnualVariableOperatingCost'), 
                                result_pack.get_df('DiscountedTechnologyEmissionsPenalty'),  
                                scene)

In [ ]:
# Evis.plot_fossil_imports(result_pack.get_df('ProductionByTechnologyAnnual'), scene)


In [ ]:
prod=result_pack.get_df('ProductionByTechnologyAnnual')
use=result_pack.get_df('UseByTechnology')
# Evis.plot_nexus_sankey(prod, use, 2021, scene)
# fig1 = Evis.plot_nexus_sankey_slider(prod, use, scenario='Base_CNZ_noCCS', step=5)
# fig2 = Evis.plot_nexus_sankey_slider(prod, use, years=[2030, 2040, 2050])
fig3 = Evis.plot_nexus_sankey_slider(prod, use, step=1)              # all 30 years

In [ ]:
fig3

# <span style="color: Coral;"> Playground for Solved Model </span>

* <span style="color: grey;">Load the Gurobi Model object as `m` from `clewsRun`'s attribute `solved_model` </span>  

In [ ]:
# m.solved_model.printStats()          # Print basic stats: number of vars, constraints, etc.